- Links uteis:

1 - https://theairlab.org/alfa-dataset/ (contém informações sobre o status do vôo.)

# Explorando os dados


In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import os
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler


# Parâmetros

In [ ]:
DATA_FILE = '../data/02_intermediate/carbonZ_master_0_to_8_merged.csv'

# Funções

In [ ]:

def plot_generic_df(df, titulo_grafico="Análise de Dados"):
    # 1. Tratamento do Tempo (Eixo X)
    if '%time' in df.columns:
        t_zero = df['%time'].iloc[0]
        tempo_segundos = (df['%time'] - t_zero) / 1e9
    else:
        print("Coluna %time não encontrada. Usando índice como tempo.")
        tempo_segundos = df.index

    # 2. Filtrar apenas colunas numéricas (e excluir metadados de tempo/sequência)
    cols_numericas = df.select_dtypes(include=['number']).columns.tolist()
    # Ignoramos colunas que são apenas contadores ou carimbos de tempo para o plot ficar limpo
    blacklist = ['%time', 'field.header.stamp', 'field.header.seq']
    cols_para_plotar = [c for c in cols_numericas if c not in blacklist]

    # 3. Configurar Subplots dinamicamente
    # Vamos agrupar as colunas para não criar 20 subplots (ex: máximo de 3 colunas por gráfico)
    num_cols = len(cols_para_plotar)
    if num_cols == 0:
        print("Nenhuma coluna numérica encontrada para plotar.")
        return

    # Criamos subplots: um para cada 3 variáveis (ou conforme sua preferência)
    v_limit = 4 # variáveis por subplot
    n_subplots = (num_cols + v_limit - 1) // v_limit
    
    fig, axes = plt.subplots(n_subplots, 1, figsize=(12, 4 * n_subplots), sharex=True)
    
    # Se houver apenas 1 subplot, o matplotlib não retorna uma lista, então ajustamos:
    if n_subplots == 1: axes = [axes]

    # 4. Plotagem em massa
    for i in range(n_subplots):
        start = i * v_limit
        end = start + v_limit
        batch_cols = cols_para_plotar[start:end]
        
        for col in batch_cols:
            # Pegamos apenas o final do nome da coluna (ex: 'linear.x') para a legenda
            label_curto = col.split('.')[-1]
            axes[i].plot(tempo_segundos, df[col], label=label_curto)
        
        axes[i].legend(loc='upper right', fontsize='small')
        axes[i].grid(True, alpha=0.3)
        axes[i].set_ylabel("Valores")

    axes[-1].set_xlabel("Tempo (segundos)")
    plt.suptitle(titulo_grafico, fontsize=14)
    plt.tight_layout(rect=[0, 0.03, 1, 0.97])
    plt.show()

# Exemplo de uso para o seu primeiro DF:
# plot_generic_df(df_primeiro, titulo_grafico=primeira_chave)

# Abrindo arquivo



In [ ]:
df = pd.read_csv(DATA_FILE)


In [ ]:
df

# Descartar Colunas de Ruído

Remove-se metadados de protocolo, IDs de sistema e campos brutos do MAVLink que não possuem valor físico para o aprendizado de máquina.

In [ ]:
# Lista de padrões de nomes que representam ruído/metadados
keywords_ruido = [
    'header', 'seq', 'stamp', 'frame_id', 'magic', 'len', 
    'flags', 'sysid', 'compid', 'msgid', 'checksum', 'payload'
]

# Filtrar as colunas que NÃO contêm as palavras-chave acima
cols_ruido = [col for col in df.columns if any(key in col for key in keywords_ruido)]

# Criar dataframe sem ruído (Limpando o excesso)
df_limpo = df.drop(columns=cols_ruido)

print(f"Colunas removidas: {len(cols_ruido)}")
print(f"Colunas restantes: {len(df_limpo.columns)}")

In [ ]:
df_limpo

## Renomeando colunas

In [ ]:
# Dicionário de mapeamento: Nome Original -> Nome Intuitivo
mapeamento_nomes = {
    'field.twist.linear.x': 'vel_gps_x',
    'field.twist.linear.y': 'vel_gps_y',
    'field.twist.linear.z': 'vel_gps_z',
    'field.twist.angular.x': 'taxa_giro_roll',
    'field.twist.angular.y': 'taxa_giro_pitch',
    'field.twist.angular.z': 'taxa_giro_yaw',
    'engine_status': 'status_motor_log',
    'rc_aileron': 'comando_aileron',
    'rc_elevator': 'comando_profundor',
    'rc_throttle': 'comando_acelerador',
    'rc_rudder': 'comando_leme',
    'rc_signal_strength': 'forca_sinal_rc',
    'field.status.status': 'status_gps_fix',
    'field.latitude': 'latitude',
    'field.longitude': 'longitude',
    'field.altitude': 'altitude_msl',
    'field.position_covariance0': 'incerteza_gps_horizontal',
    'field.angular_velocity.x': 'vel_angular_x',
    'field.angular_velocity.y': 'vel_angular_y',
    'field.angular_velocity.z': 'vel_angular_z',
    'field.linear_acceleration.x': 'aceleracao_x_frontal',
    'field.linear_acceleration.y': 'aceleracao_y_lateral',
    'field.linear_acceleration.z': 'aceleracao_z_vertical',
    'field.linear_acceleration_covariance0': 'ruido_acelerometro_x',
    'field.linear_acceleration_covariance4': 'ruido_acelerometro_y',
    'field.linear_acceleration_covariance8': 'ruido_acelerometro_z'
}

# Renomeando no DataFrame
df_limpo = df_limpo.rename(columns=mapeamento_nomes)

# Verificando os novos nomes
print("Colunas renomeadas:")
print(df_limpo.columns.tolist())

# Limpeza extra

In [ ]:
df_limpo = df_limpo.drop_duplicates()

df_limpo = df_limpo.dropna(subset=['%time'])

# Converter %time para segundos relativos 
# Isso faz o tempo começar em 0
df_limpo['tempo_relativo'] = (df_limpo['%time'] - df_limpo['%time'].iloc[0]) / 1e9 

print(f"Dataset limpo: {df_limpo.shape[0]} linhas prontas.")

# Contagem de nulos por coluna

In [ ]:
# Contagem total de nulos por coluna
nulos_por_coluna = df_limpo.isnull().sum()

# Filtrar apenas as colunas que possuem pelo menos um nulo para facilitar a leitura
colunas_com_nulos = nulos_por_coluna[nulos_por_coluna > 0]

preencher_nans = False

print("Contagem de valores nulos antes do tratamento:")
if not colunas_com_nulos.empty:
    print(colunas_com_nulos)
    if preencher_nans:
        # Usamos ffill (preencher com o anterior) e bfill (preencher com o próximo) 
        # para garantir que não restem valores nulos após o merge
        df_limpo = df_limpo.ffill().bfill()
else:
    print("Nenhum valor nulo encontrado!")

# Normalização

Usa-se o MinMaxScaler que é excelente para Redes Neurais e Autoencoders, pois garante que nenhum valor escape do intervalo definido..

In [ ]:
# 1. Selecionar colunas (exceto tempo)
cols_treino = [c for c in df_limpo.columns if c not in ['%time', 'tempo_relativo']]

# 2. Aplicar MinMaxScaler (Garante intervalo entre 0 e 1)
scaler_minmax = MinMaxScaler(feature_range=(0, 1))
df_norm = df_limpo.copy()

# Forçamos a conversão para float para evitar erros de tipo
df_norm[cols_treino] = scaler_minmax.fit_transform(df_limpo[cols_treino].astype(float))

# 3. VALIDAÇÃO: Agora o máximo TEM que ser 1 e o mínimo TEM que ser 0
print("--- Verificação de Intervalo Fixo ---")
print(f"Valor Máximo: {df_norm[cols_treino].max().max()}")
print(f"Valor Mínimo: {df_norm[cols_treino].min().min()}")

In [ ]:
output_dir = '../data/03_primary'

# Salvar o dataframe completo (Master)
file_path = os.path.join(output_dir, 'carbonZ_master_normalizado_sem_colunas_ruido.csv')
df_norm.to_csv(file_path, index=False)

print(f"Dataset mestre salvo com sucesso em: {file_path}")

# Primeira filtragem de features para modelagem

Foca-se nas colunas que descrevem a física do voo e a intenção do piloto, essenciais para o modelo semi-supervisionado detectar anomalias.

In [ ]:
# Seleção estratégica de colunas para Detecção de Falhas (Engine Failure)
features_treino = [
    '%time',                   # Essencial para séries temporais
    'rc_throttle',             # Causa (Comando)
    'rc_elevator',             # Causa (Comando)
    'rc_aileron',              # Causa (Comando)
    'field.linear_acceleration.x', # Efeito imediato (Física)
    'field.angular_velocity.y',    # Efeito imediato (Atitude/Pitch)
    'vel_medida_x',            # Estado atual
    'erro_vel_x',              # Feature de Ouro (Desvio de performance)
    'altitude_relativa',       # Estado atual (Sustentação)
    'field.twist.linear.z',    # Taxa de subida/descida real
    'field.linear_acceleration_covariance0', # Saúde do sensor/Vibração
    'field.position_covariance0'             # Integridade do GPS (Útil para ataques)
]

# Criar o DataFrame final de treinamento
df_modelo = df_limpo[features_treino].copy()

# Opcional: Remover os primeiros NaNs (delay de inicialização dos sensores)
df_modelo = df_modelo.dropna().reset_index(drop=True)

print("Colunas selecionadas para o modelo:")
print(df_modelo.columns.tolist())

In [ ]:
df_modelo.shape